# 02 — Coordinate Projection

## The Problem with Angular Coordinates

Latitude and longitude are angular measurements, not distances. One degree of longitude at 80°N spans approximately 19 km of ground, while the same degree at the equator spans approximately 111 km. If we tried to build a uniform-area grid directly in (lat, lon) space, tiles near the poles would be far smaller in ground area than tiles at the equator — making the 250 m tile size meaningless at high latitudes.

## Solution: Web Mercator (EPSG:3857)

The Web Mercator projection maps (lat, lon) to (x, y) in metres using:

$$x = R \cdot \lambda$$

$$y = R \cdot \ln\!\left(\tan\!\left(\frac{\pi}{4} + \frac{\phi}{2}\right)\right)$$

where $R$ = 6 378 137 m (WGS84 equatorial radius), $\lambda$ = longitude in radians, $\phi$ = latitude in radians.

The logarithmic term — the inverse Gudermannian — makes the projection **conformal** (locally angle-preserving), which is why it is used in web mapping. However, it also compresses high-latitude areas, causing **scale distortion** (a 250 m Mercator tile covers more real ground near the poles).

The projection is undefined at the poles because $\tan(\pi/2)$ diverges; by convention, latitudes are clamped to ±85.05°.

In [1]:
import math
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from map_encryption import _project, _unproject, _R_EARTH, _HALF_WORLD, SchemeParams

bin_size_m = SchemeParams().bin_size_m
M = 2 * int(math.floor(_HALF_WORLD / bin_size_m)) + 1
print(f'Half-world extent: {_HALF_WORLD:.3f} m')
print(f'Bin size: {bin_size_m} m')
print(f'Total grid cells per axis: M = {M}')
print(f'Total cells: M^2 = {M**2:,}')

Half-world extent: 20037508.343 m
Bin size: 250 m
Total grid cells per axis: M = 160301
Total cells: M^2 = 25,696,410,601


In [2]:
cities = [
    ('Times Square',  40.758,  -73.985),
    ('London',        51.507,   -0.128),
    ('Tokyo',         35.689,  139.692),
    ('Sydney',       -33.868,  151.209),
    ('Oslo',          59.913,   10.752),
    ('Nairobi',       -1.286,   36.818),
    ('Sao Paulo',    -23.551,  -46.633),
    ('Singapore',      1.352,  103.820),
]

print(f'{"City":<15} {"Lat":>8} {"Lon":>9} {"x (m)":>15} {"y (m)":>15}')
print('-' * 65)
for name, lat, lon in cities:
    x, y = _project(lat, lon)
    rlat, rlon = _unproject(x, y)
    err = max(abs(rlat - lat), abs(rlon - lon))
    assert err < 1e-10, f'Round-trip error {err} exceeds tolerance for {name}'
    print(f'{name:<15} {lat:>8.3f} {lon:>9.3f} {x:>15.3f} {y:>15.3f}')

print('\nAll round-trips verified.')

City                 Lat       Lon           x (m)           y (m)
-----------------------------------------------------------------
Times Square      40.758   -73.985    -8235972.526     4976711.982
London            51.507    -0.128      -14248.895     6711470.935
Tokyo             35.689   139.692    15550442.308     4257912.202
Sydney           -33.868   151.209    16832508.883    -4011091.393
Oslo              59.913    10.752     1196907.165     8380393.718
Nairobi           -1.286    36.818     4098561.012     -143168.886
Sao Paulo        -23.551   -46.633    -5191161.814    -2698790.173
Singapore          1.352   103.820    11557189.534      150517.921

All round-trips verified.


In [3]:
lats_range = np.linspace(-80, 80, 400)
y_vals = [_project(lat, 0)[1] for lat in lats_range]
scale_factors = [math.cos(math.radians(lat)) for lat in lats_range]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=lats_range, y=y_vals, name='y_mercator (m)',
    line=dict(color='steelblue', width=2)
))
fig.add_trace(go.Scatter(
    x=lats_range, y=[s * max(y_vals) for s in scale_factors],
    name='scale factor × y_max (normalised)',
    line=dict(color='tomato', width=2, dash='dash'),
    yaxis='y2'
))
fig.update_layout(
    title='Web Mercator y-coordinate and scale factor vs latitude',
    xaxis_title='Latitude (degrees)',
    yaxis_title='y_mercator (m)',
    yaxis2=dict(title='Scale factor (cos lat)', overlaying='y', side='right'),
    template='plotly_white',
    height=400,
    legend=dict(x=0.02, y=0.98)
)
fig.show()

## Grid Dimensions

With `bin_size_m = 250` m and `_HALF_WORLD ≈ 20 037 508` m:

```
M = 2 * floor(20_037_508 / 250) + 1 = 160_301 cells per axis
Total cells = M² = 25_696_411_601 ≈ 25.7 billion
```

Each tile index (qx, qy) is a pair of integers in [-80150, +80150]. Even though the Mercator projection distorts scale at high latitudes, the **index space** is uniform: every (qx, qy) pair is exactly one 250 m Mercator-unit tile. The PRP shuffles these indices, so an attacker who sees encrypted (qxp, qyp) values cannot determine which region of the globe they came from.

In [4]:
import pandas as pd

rows = []
for name, lat, lon in cities:
    x, y = _project(lat, lon)
    rlat, rlon = _unproject(x, y)
    rows.append({'city': name, 'orig_lat': lat, 'orig_lon': lon,
                 'recovered_lat': rlat, 'recovered_lon': rlon, 'type': 'original'})
    rows.append({'city': name, 'orig_lat': rlat, 'orig_lon': rlon,
                 'recovered_lat': rlat, 'recovered_lon': rlon, 'type': 'recovered'})

df = pd.DataFrame(rows)

fig = px.scatter_mapbox(
    df, lat='orig_lat', lon='orig_lon', color='type',
    hover_name='city',
    mapbox_style='open-street-map',
    zoom=1, center={'lat': 20, 'lon': 0},
    title='Round-trip fidelity: project → unproject for 8 world cities',
    height=500,
    color_discrete_map={'original': 'steelblue', 'recovered': 'tomato'}
)
fig.show()

/var/folders/gl/ppzrtfy95t7ghxlyd9tf3vk00000gn/T/ipykernel_2372/2928421333.py:14: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


In [5]:
lat_samples = [0, 20, 40, 60, 80]
ground_km = [250 / math.cos(math.radians(lat)) / 1000 for lat in lat_samples]

fig = px.bar(
    x=[str(lat) + '°' for lat in lat_samples],
    y=ground_km,
    labels={'x': 'Latitude', 'y': 'Effective ground size (km)'},
    title='Effective ground size of a 250 m Mercator tile by latitude',
    template='plotly_white',
    height=400,
    color=ground_km,
    color_continuous_scale='Oranges'
)
fig.update_layout(showlegend=False, coloraxis_showscale=False)
fig.show()

## What the Projection Gives — and Does Not Give

**Gives:** uniform index space (every cell is 250 m × 250 m in Mercator units), global coverage (±85.05° latitude), full invertibility, and conformality (locally angle-preserving, useful for short-range distance calculations).

**Does not give:** equal-area (a 250 m tile covers more ground at high latitudes), accuracy above ±85.05° (pole singularity), or metric accuracy at continental scales.

NB03 builds the tile grid and the Feistel PRP on top of this projection.